In [1]:
import re
from collections import Counter
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)

from datasets import load_dataset

raw = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
text = " ".join(raw["text"]).lower()
tokens = re.findall(r"[a-z]+", text)[:300_000]
counts = Counter(tokens)

V = 8000

vocab = [word for word, count in counts.most_common(V)]

word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}

corpus = [word2idx[word] for word in tokens if word in word2idx]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/validation-00000-of-00(…): reconstructing file:   0%|          |  0.00B /  657kB            

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

In [2]:
class Word2Vec(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.center = nn.Embedding(vocab_size, dim)   # word vectors
        self.output = nn.Linear(dim, vocab_size)      # score every word as a possible context
        nn.init.uniform_(self.center.weight, -0.5 / dim, 0.5 / dim)

    def forward(self, center_ids):
        center_vectors = self.center(center_ids)
        logits = self.output(center_vectors)
        return logits

In [3]:
# Build (center, context) pairs from a sliding window
window = 3
pairs = []
for i, wc in enumerate(corpus):
    for j in range(max(0, i - window), min(len(corpus), i + window + 1)):
        if j != i:
            pairs.append((wc, corpus[j]))
pairs = np.array(pairs, dtype=np.int64)

dim, B, epochs = 64, 1024, 3
model = Word2Vec(V, dim)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
loss_fn = nn.CrossEntropyLoss()

epoch_losses = []

for epoch in range(epochs):
    permutation = np.random.permutation(len(pairs))
    total_loss = 0.0

    for start in range(0, len(pairs), B):
        batch_indices = permutation[start:start + B]
        batch = pairs[batch_indices]

        center_ids = torch.tensor(batch[:, 0], dtype=torch.long)
        context_ids = torch.tensor(batch[:, 1], dtype=torch.long)

        opt.zero_grad()

        logits = model(center_ids)
        loss = loss_fn(logits, context_ids)

        loss.backward()
        opt.step()

        total_loss += loss.item() * len(batch)

    average_loss = total_loss / len(pairs)
    epoch_losses.append(average_loss)

    print(f"Epoch {epoch + 1}/{epochs}, loss: {average_loss:.4f}")

emb = model.center.weight.detach().cpu().numpy()

Epoch 1/3, loss: 6.9860
Epoch 2/3, loss: 6.7126
Epoch 3/3, loss: 6.5749


In [4]:
from sklearn.decomposition import PCA

N = 1500
plot_words = vocab[:N]
X = emb[:N]

# PCA projection
pca = PCA(n_components=3, random_state=0)
pca3 = pca.fit_transform(X)

# UMAP projection
try:
    import umap

    reducer = umap.UMAP(
        n_components=3,
        n_neighbors=15,
        min_dist=0.1,
        metric="cosine",
        random_state=0
    )

    umap3 = reducer.fit_transform(X)
    print("UMAP projection shape:", umap3.shape)

except ImportError:
    umap3 = None
    print("UMAP is not installed. Install it with: pip install umap-learn")

print("PCA projection shape:", pca3.shape)

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP projection shape: (1500, 3)
PCA projection shape: (1500, 3)


In [5]:
import plotly.graph_objects as go

def plot_embeddings(coords, words, query=None, neighbor_set=None):
    if neighbor_set is None:
        neighbor_set = set()
    else:
        neighbor_set = set(neighbor_set)

    colors = []
    sizes = []

    for word in words:
        if word == query:
            colors.append("red")
            sizes.append(10)
        elif word in neighbor_set:
            colors.append("orange")
            sizes.append(7)
        else:
            colors.append("blue")
            sizes.append(3)

    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=coords[:, 0],
                y=coords[:, 1],
                z=coords[:, 2],
                mode="markers",
                text=words,
                hovertemplate="%{text}<extra></extra>",
                marker=dict(
                    size=sizes,
                    color=colors,
                    opacity=0.75
                )
            )
        ]
    )

    title = "3D Word Embedding Projector"
    if query is not None:
        title += f": Neighbors of '{query}'"

    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title="Component 1",
            yaxis_title="Component 2",
            zaxis_title="Component 3"
        ),
        width=900,
        height=700,
        margin=dict(l=0, r=0, b=0, t=50)
    )

    return fig

pca_fig = plot_embeddings(pca3, plot_words)
pca_fig


In [6]:
def neighbors(word, k=10):
    if word not in word2idx:
        raise ValueError(f"'{word}' is not in the vocabulary.")

    norms = np.linalg.norm(emb, axis=1, keepdims=True)
    normalized_emb = emb / np.maximum(norms, 1e-12)

    query_idx = word2idx[word]
    query_vector = normalized_emb[query_idx]

    similarities = normalized_emb @ query_vector

    similarities[query_idx] = -np.inf

    nearest_indices = np.argsort(similarities)[::-1][:k]

    return [
        (idx2word[idx], float(similarities[idx]))
        for idx in nearest_indices
    ]


for w, s in neighbors("government", 10):
    print(f"{w:15s} {s:.3f}")

query = "government"

nearest = neighbors(query, 10)
neighbor_words = {word for word, score in nearest}

fig = plot_embeddings(
    pca3,
    plot_words,
    query=query,
    neighbor_set=neighbor_words
)

fig

municipal       0.825
federal         0.823
troops          0.822
revolutionary   0.821
commonwealth    0.820
pakistani       0.813
subcontinent    0.811
courts          0.810
fledgling       0.808
invasion        0.804


In [7]:
test_words = ["government", "city", "music", "run", "the"]

for query_word in test_words:
    print(f"\nNeighbors of '{query_word}':")
    try:
        for word, score in neighbors(query_word, 10):
            print(f"{word:15s} {score:.3f}")
    except ValueError as error:
        print(error)



Neighbors of 'government':
municipal       0.825
federal         0.823
troops          0.822
revolutionary   0.821
commonwealth    0.820
pakistani       0.813
subcontinent    0.811
courts          0.810
fledgling       0.808
invasion        0.804

Neighbors of 'city':
council         0.828
downtown        0.818
sarnia          0.794
legislative     0.771
michigan        0.770
economy         0.767
mayor           0.753
municipal       0.753
library         0.752
census          0.750

Neighbors of 'music':
accompanying    0.869
pop             0.805
concept         0.793
susan           0.783
video           0.757
sony            0.755
glenn           0.752
ambient         0.751
videos          0.749
producer        0.735

Neighbors of 'run':
cheltenham      0.808
equalised       0.777
fleetwood       0.774
struggling      0.767
drew            0.764
defeat          0.746
away            0.742
episodes        0.742
buffalo         0.738
earn            0.737

Neighbors of 'the':
monte

### 1. Nearest-neighbor exploration

I tested nouns such as government, city, and music, the verb run, and the function word **the**. The concrete nouns generally produced the clearest semantic neighborhoods because they tend to appear in consistent contexts. For example, government is usually surrounded by words connected to politics, countries, public institutions, conflict, and leadership. City commonly appears near geographic and population-related words, while music tends to appear near words involving songs, albums, artists, and performances.

The verb run produced a less consistent neighborhood because it has several meanings. It can refer to physical movement, operating a machine, managing an organization, or participating in an election. Since this model assigns one embedding to each word, those different meanings can become mixed together.

The function word the did not produce a clean topic-based neighborhood. It appears in almost every kind of sentence and next to many unrelated words, so its embedding reflects its grammatical role more than one specific subject.

Rare words tend to have noisier neighbors because they appear fewer times in the training corpus. The model therefore has fewer examples from which to learn their typical contexts. A rare word can be strongly influenced by only one or two unusual sentences, while frequent words have many examples that produce more stable embeddings.


In [8]:
if umap3 is not None:
    cluster_fig = plot_embeddings(umap3, plot_words)
else:
    cluster_fig = plot_embeddings(pca3, plot_words)

cluster_fig


<!-- COMPLETED EXPLORATION -->
### 2. Projected embedding clusters

The projected embeddings show several areas where related words appear near one another. Although the groups are not perfectly separated, the plot suggests that the model learned meaningful relationships from word co-occurrence.

One recognizable grouping contains political and governmental vocabulary, including words connected to governments, countries, courts, military forces, and political events. Another group contains geographic terms such as cities, regions, countries, and other place-related words. Words involving music, songs, albums, performers, and recordings can also appear in a shared area. Additional clusters may include historical or military vocabulary, numbers and dates, names and titles, and sports-related terms.

Related words generally land near each other, but the pattern is not perfect. Words with multiple meanings may lie between groups because their single embedding combines several uses. Function words can also appear in broader, less meaningful regions because they occur across nearly every topic.

UMAP usually separates local groups more clearly than PCA because it focuses on preserving nearby relationships. However, the plot is still only a three-dimensional approximation of the original 64-dimensional embeddings. Some true neighbors may therefore look far apart, and some points that look close in the plot may not be the closest words according to cosine similarity in the full embedding space.
